# Foreground Validation Metrics

Compute PSNR, SSIM, and LPIPS for validation renders using foreground masks. PSNR is computed on masked pixels only. SSIM and LPIPS are computed on the masked foreground crop after setting pixels outside the mask to the same fill value in both images. Set `MASK_ERODE_PIXELS` to remove boundary pixels from the mask before metrics are computed.

The defaults target the sedan validation result folders in this repo and the sedan masks prepared in `prepare_sedan_nobg.ipynb`. If your masks live somewhere else, change `MASK_DIR` in the config cell; the notebook also checks common mask folders under each result directory.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import warnings
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

try:
    import lpips
except ImportError as exc:
    lpips = None
    LPIPS_IMPORT_ERROR = exc
else:
    LPIPS_IMPORT_ERROR = None

# Override with FOREGROUND_METRICS_RESULT_DIRS="dir_a:dir_b" if needed.
RESULT_DIRS = [
    Path("data/result/envgs_sedan_mast3r"),
    Path("data/result/sedan_no_bg_01_spec"),
]
if os.environ.get("FOREGROUND_METRICS_RESULT_DIRS"):
    RESULT_DIRS = [Path(p) for p in os.environ["FOREGROUND_METRICS_RESULT_DIRS"].split(":") if p]

# This is the mask folder used by prepare_sedan_nobg.ipynb.
MASK_DIR = Path(os.environ.get("SEDAN_MASK_DIR", "/home/roman/ba/sedan_car_masks"))

OUTPUT_NAME = os.environ.get("FOREGROUND_METRICS_OUTPUT", "foreground_metrics.json")
WRITE_OUTPUTS = os.environ.get("FOREGROUND_METRICS_WRITE_OUTPUTS", "1") != "0"
MAX_IMAGES_ENV = os.environ.get("FOREGROUND_METRICS_MAX_IMAGES")
MAX_IMAGES = int(MAX_IMAGES_ENV) if MAX_IMAGES_ENV else None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LPIPS_NET = "vgg"
COMPUTE_LPIPS = True

MASK_THRESHOLD = 127
MASK_ERODE_PIXELS = 10 #int(os.environ.get("FOREGROUND_METRICS_ERODE_PIXELS", "0"))
MASK_REJECT_COVERAGE = 0.999
BBOX_PAD = 0
FILL_VALUE = 0.0
SSIM_WINDOW_SIZE = 11

print("result dirs:", [str(p) for p in RESULT_DIRS])
print("mask dir:", MASK_DIR)
print("mask erosion pixels:", MASK_ERODE_PIXELS)
print("device:", DEVICE)


result dirs: ['data/result/envgs_sedan_mast3r', 'data/result/sedan_no_bg_01_spec']
mask dir: /home/roman/ba/sedan_car_masks
mask erosion pixels: 0
device: cuda


In [3]:
IMAGE_RE = re.compile(r"frame(?P<frame>\d+)_camera(?P<camera>\d+)$")
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}


def read_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return arr / 255.0


def resize_rgb_to(path: Path, size_wh: tuple[int, int]) -> np.ndarray:
    with Image.open(path) as im:
        im = im.convert("RGB")
        if im.size != size_wh:
            im = im.resize(size_wh, Image.Resampling.BILINEAR)
        arr = np.asarray(im, dtype=np.float32)
    return arr / 255.0


def erode_mask(mask: np.ndarray, pixels: int) -> np.ndarray:
    if pixels < 0:
        raise ValueError(f"MASK_ERODE_PIXELS must be >= 0, got {pixels}")
    if pixels == 0:
        return mask

    # Square-kernel binary erosion: a pixel remains foreground only if all
    # pixels within `pixels` of it were foreground in the original mask.
    kernel_size = 2 * pixels + 1
    inv = 1.0 - torch.from_numpy(mask.astype(np.float32))[None, None]
    eroded = F.max_pool2d(inv, kernel_size=kernel_size, stride=1, padding=pixels) < 0.5
    return eroded[0, 0].cpu().numpy()


def read_mask(path: Path, target_hw: tuple[int, int]) -> np.ndarray:
    target_h, target_w = target_hw
    with Image.open(path) as im:
        im = im.convert("L")
        if im.size != (target_w, target_h):
            im = im.resize((target_w, target_h), Image.Resampling.BILINEAR)
        arr = np.asarray(im)
    mask = arr > MASK_THRESHOLD
    return erode_mask(mask, MASK_ERODE_PIXELS)


def render_pairs(result_dir: Path) -> Iterator[dict]:
    render_dir = result_dir / "RENDER"
    if not render_dir.is_dir():
        raise FileNotFoundError(f"Missing RENDER folder: {render_dir}")

    pred_paths = sorted(
        p for p in render_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and not p.stem.endswith("_gt")
        and not p.stem.endswith("_error")
    )

    for pred_path in pred_paths:
        match = IMAGE_RE.match(pred_path.stem)
        if not match:
            warnings.warn(f"Skipping unrecognized render name: {pred_path.name}")
            continue

        gt_path = pred_path.with_name(f"{pred_path.stem}_gt{pred_path.suffix}")
        if not gt_path.exists():
            warnings.warn(f"Skipping {pred_path.name}: missing GT image {gt_path.name}")
            continue

        yield {
            "frame": int(match.group("frame")),
            "camera": int(match.group("camera")),
            "pred_path": pred_path,
            "gt_path": gt_path,
        }


def candidate_mask_paths(result_dir: Path, frame: int, camera: int) -> Iterator[Path]:
    names = [
        f"{camera:04d}.png",
        f"{camera:04d}.jpg",
        f"frame{frame:04d}_camera{camera:04d}.png",
        f"frame{frame:04d}_camera{camera:04d}.jpg",
        f"frame{frame:04d}_camera{camera:04d}_gt.png",
        f"frame{frame:04d}_camera{camera:04d}_gt.jpg",
    ]
    roots = [
        MASK_DIR,
        result_dir / "MASK",
        result_dir / "MASKS",
        result_dir / "MSK",
        result_dir / "masks",
    ]

    seen = set()
    for root in roots:
        for name in names:
            path = root / name
            if path not in seen:
                seen.add(path)
                yield path

    for path in [
        result_dir / "ALPHA" / f"frame{frame:04d}_camera{camera:04d}_gt.png",
        result_dir / "ALPHA" / f"frame{frame:04d}_camera{camera:04d}.png",
    ]:
        if path not in seen:
            yield path


def resolve_mask(result_dir: Path, frame: int, camera: int, target_hw: tuple[int, int]) -> tuple[np.ndarray, Path]:
    skipped = []
    for path in candidate_mask_paths(result_dir, frame, camera):
        if not path.exists():
            continue
        mask = read_mask(path, target_hw)
        pixels = int(mask.sum())
        coverage = pixels / mask.size
        if pixels == 0:
            skipped.append(f"{path} was empty")
            continue
        if coverage >= MASK_REJECT_COVERAGE:
            skipped.append(f"{path} covered {coverage:.3f} of the image")
            continue
        return mask, path

    details = "; ".join(skipped) if skipped else "no candidate mask files existed"
    raise FileNotFoundError(f"No usable mask for frame={frame} camera={camera}: {details}")


def mask_bbox(mask: np.ndarray, pad: int = 0) -> tuple[int, int, int, int]:
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if not len(rows) or not len(cols):
        raise ValueError("Mask is empty")
    h, w = mask.shape
    y0 = max(int(rows[0]) - pad, 0)
    y1 = min(int(rows[-1]) + pad + 1, h)
    x0 = max(int(cols[0]) - pad, 0)
    x1 = min(int(cols[-1]) + pad + 1, w)
    return y0, y1, x0, x1


def masked_spatial_crop(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, tuple[int, int, int, int]]:
    pred_masked = pred.copy()
    gt_masked = gt.copy()
    pred_masked[~mask] = FILL_VALUE
    gt_masked[~mask] = FILL_VALUE
    y0, y1, x0, x1 = mask_bbox(mask, BBOX_PAD)
    return pred_masked[y0:y1, x0:x1], gt_masked[y0:y1, x0:x1], (x0, y0, x1 - x0, y1 - y0)


In [4]:
def psnr_masked(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray, eps: float = 1e-10) -> float:
    if not mask.any():
        return float("nan")
    diff = pred[mask] - gt[mask]
    mse = float(np.mean(diff * diff))
    return 10.0 * math.log10(1.0 / max(mse, eps))


def _ssim_window(window_size: int, channels: int, *, device: str, dtype: torch.dtype) -> torch.Tensor:
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    gauss = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    gauss = gauss / gauss.sum()
    window = torch.outer(gauss, gauss).view(1, 1, window_size, window_size)
    return window.expand(channels, 1, window_size, window_size).contiguous()


@torch.inference_mode()
def ssim_torch(pred: np.ndarray, gt: np.ndarray, window_size: int = SSIM_WINDOW_SIZE) -> float:
    if min(pred.shape[:2]) < window_size:
        warnings.warn(f"SSIM crop is smaller than window size {window_size}: {pred.shape[:2]}")
        return float("nan")

    x = torch.from_numpy(pred).permute(2, 0, 1)[None].to(DEVICE).float()
    y = torch.from_numpy(gt).permute(2, 0, 1)[None].to(DEVICE).float()
    channels = x.shape[1]
    weight = _ssim_window(window_size, channels, device=DEVICE, dtype=x.dtype)
    padding = window_size // 2

    mu_x = F.conv2d(x, weight, padding=padding, groups=channels)
    mu_y = F.conv2d(y, weight, padding=padding, groups=channels)
    mu_x_sq = mu_x.pow(2)
    mu_y_sq = mu_y.pow(2)
    mu_xy = mu_x * mu_y

    var_x = F.conv2d(x * x, weight, padding=padding, groups=channels) - mu_x_sq
    var_y = F.conv2d(y * y, weight, padding=padding, groups=channels) - mu_y_sq
    cov_xy = F.conv2d(x * y, weight, padding=padding, groups=channels) - mu_xy

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    ssim_map = ((2 * mu_xy + c1) * (2 * cov_xy + c2)) / ((mu_x_sq + mu_y_sq + c1) * (var_x + var_y + c2))
    return float(ssim_map.mean().item())


class LPIPSScorer:
    def __init__(self, net: str = LPIPS_NET, device: str = DEVICE):
        if lpips is None:
            raise ImportError("lpips is not installed") from LPIPS_IMPORT_ERROR
        self.device = device
        self.model = lpips.LPIPS(net=net, verbose=False).to(device).eval()

    @torch.inference_mode()
    def __call__(self, pred: np.ndarray, gt: np.ndarray) -> float:
        # Keep normalize=False to match this repo's existing metric_utils.lpips behavior.
        x = torch.from_numpy(pred).permute(2, 0, 1)[None].to(self.device).float()
        y = torch.from_numpy(gt).permute(2, 0, 1)[None].to(self.device).float()
        return float(self.model(x, y).mean().item())


lpips_scorer = LPIPSScorer() if COMPUTE_LPIPS else None


/home/roman/ba/envgs/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/roman/ba/envgs/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
def clean_float(value):
    if value is None or pd.isna(value):
        return None
    return float(value)


def summarize_df(df: pd.DataFrame) -> dict:
    summary = {"num_images": int(len(df))}
    for metric in ["psnr", "ssim", "lpips"]:
        if metric in df:
            summary[f"{metric}_mean"] = clean_float(df[metric].mean())
            summary[f"{metric}_std"] = clean_float(df[metric].std(ddof=0))
    for metric in ["mask_pixels", "mask_coverage"]:
        if metric in df:
            summary[f"{metric}_mean"] = clean_float(df[metric].mean())
    return summary


def evaluate_result_dir(result_dir: Path) -> pd.DataFrame:
    pairs = list(render_pairs(result_dir))
    if MAX_IMAGES is not None:
        pairs = pairs[:MAX_IMAGES]

    rows = []
    for pair in tqdm(pairs, desc=result_dir.name):
        pred_path = pair["pred_path"]
        gt_path = pair["gt_path"]
        pred = read_rgb(pred_path)
        h, w = pred.shape[:2]
        gt = resize_rgb_to(gt_path, (w, h))
        mask, mask_path = resolve_mask(result_dir, pair["frame"], pair["camera"], (h, w))
        pred_crop, gt_crop, bbox_xywh = masked_spatial_crop(pred, gt, mask)

        rows.append({
            "result_dir": str(result_dir),
            "image": pred_path.name,
            "frame": pair["frame"],
            "camera": pair["camera"],
            "psnr": psnr_masked(pred, gt, mask),
            "ssim": ssim_torch(pred_crop, gt_crop),
            "lpips": lpips_scorer(pred_crop, gt_crop) if lpips_scorer is not None else float("nan"),
            "mask_pixels": int(mask.sum()),
            "mask_coverage": float(mask.mean()),
            "bbox_xywh": list(map(int, bbox_xywh)),
            "pred_path": str(pred_path),
            "gt_path": str(gt_path),
            "mask_path": str(mask_path),
        })
    return pd.DataFrame(rows)


def write_metrics(result_dir: Path, df: pd.DataFrame, summary: dict) -> None:
    payload = {
        "config": {
            "result_dir": str(result_dir),
            "mask_dir": str(MASK_DIR),
            "mask_threshold": MASK_THRESHOLD,
            "mask_erode_pixels": MASK_ERODE_PIXELS,
            "mask_reject_coverage": MASK_REJECT_COVERAGE,
            "bbox_pad": BBOX_PAD,
            "fill_value": FILL_VALUE,
            "ssim_window_size": SSIM_WINDOW_SIZE,
            "lpips_net": LPIPS_NET if COMPUTE_LPIPS else None,
            "lpips_normalize": False,
        },
        "summary": summary,
        "metrics": json.loads(df.to_json(orient="records")),
    }

    json_path = result_dir / OUTPUT_NAME
    csv_path = result_dir / OUTPUT_NAME.replace(".json", ".csv")
    json_path.write_text(json.dumps(payload, indent=4), encoding="utf-8")
    df.to_csv(csv_path, index=False)
    print(f"wrote {json_path}")
    print(f"wrote {csv_path}")


In [6]:
all_rows = []
all_summaries = []

for result_dir in RESULT_DIRS:
    if not result_dir.exists():
        warnings.warn(f"Skipping missing result dir: {result_dir}")
        continue

    df = evaluate_result_dir(result_dir)
    if df.empty:
        warnings.warn(f"No render pairs found in: {result_dir}")
        continue

    summary = summarize_df(df)
    summary["result_dir"] = str(result_dir)
    all_rows.append(df)
    all_summaries.append(summary)

    if WRITE_OUTPUTS:
        write_metrics(result_dir, df, summary)

summary_df = pd.DataFrame(all_summaries)
if not summary_df.empty:
    cols = ["result_dir", "num_images", "psnr_mean", "ssim_mean", "lpips_mean", "psnr_std", "ssim_std", "lpips_std", "mask_coverage_mean"]
    display(summary_df[[c for c in cols if c in summary_df.columns]])

    if WRITE_OUTPUTS:
        summary_path = Path("data/result/foreground_metrics_summary.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"wrote {summary_path}")
else:
    print("No metrics computed.")


envgs_sedan_mast3r:   0%|          | 0/20 [00:00<?, ?it/s]

wrote data/result/envgs_sedan_mast3r/foreground_metrics.json
wrote data/result/envgs_sedan_mast3r/foreground_metrics.csv


sedan_no_bg_01_spec:   0%|          | 0/20 [00:00<?, ?it/s]

wrote data/result/sedan_no_bg_01_spec/foreground_metrics.json
wrote data/result/sedan_no_bg_01_spec/foreground_metrics.csv


,result_dir,num_images,psnr_mean,ssim_mean,lpips_mean,psnr_std,ssim_std,lpips_std,mask_coverage_mean
0,data/result/envgs_sedan_mast3r,20,26.994236,0.876564,0.173439,2.110279,0.038304,0.027060,0.289387
1,data/result/sedan_no_bg_01_spec,20,25.117492,0.868958,0.174502,1.867004,0.038259,0.025571,0.289387


wrote data/result/foreground_metrics_summary.csv
